# Swin-UMamba trên Kaggle — Train / Val / Inference

Notebook này chạy lại pipeline của bài gốc *Swin-UMamba* (nnU-Net v2, cấu hình **2d**), nhưng dùng bộ dữ liệu mới.

**Trước khi chạy, cấu hình Kaggle:**
1. *Settings → Accelerator*: **GPU T4** (dùng 1 GPU). Không nên dùng P100 vì torch mới trên Kaggle không còn hỗ trợ tốt.
2. *Settings → Internet*: **On** (để clone GitHub, cài `mamba-ssm` và tải pretrained VMamba).
3. *Add Input*: thêm Kaggle Dataset chứa thư mục BTXRD đã chia sẵn, cấu trúc giống `data/BTXRD`:
   ```
   BTXRD_split/
   ├── split.csv          (image_id,split,type  — type = benign / malignant)
   ├── train/{images, Annotations}
   ├── val/{images, Annotations}
   └── test/{images, Annotations}
   ```
   Notebook tự tìm thư mục có `train/Annotations` trong `/kaggle/input` (hoặc đặt `BTXRD_DIR`).
   - `train` → huấn luyện, `val` → validation của nnU-Net (fold 0 dùng đúng split này), `test` → inference & đánh giá.
   - Bài toán: **segment + phân loại** mỗi pixel thành 0 = nền, 1 = u lành tính, 2 = u ác tính.
4. Nếu train lâu hơn 12 giờ: dùng **Save Version → Save & Run All (Commit)**. Kết quả trong `/kaggle/working` sẽ được lưu lại. Ở phiên sau, thêm output đó làm input và đặt `RESUME_DIR` để train tiếp.

Các bước: clone repo → cài môi trường → tải pretrained → chuyển dữ liệu sang định dạng nnU-Net → `nnUNetv2_plan_and_preprocess` → `nnUNetv2_train` (kèm validation) → `nnUNetv2_predict` trên tập test → tính DSC / NSD → trực quan hoá.

## 0. Cấu hình

In [1]:
import os, glob, json

# ---------- Mã nguồn ----------
REPO_URL = "https://github.com/67htb5vn/Swin-UMamba.git"
BRANCH   = "main"
WORK     = "/kaggle/working"
REPO     = f"{WORK}/Swin-UMamba"
TMP      = "/tmp/nnunet"            # raw + preprocessed để ở /tmp (đĩa lớn, không bị lưu vào output)

# ---------- Dữ liệu ----------
# "btxrd"     : chuyển BTXRD (LabelMe json) sang nnU-Net
# "nnunet_raw": dùng sẵn một thư mục DatasetXXX_* đúng định dạng nnU-Net (vd. dữ liệu gốc của U-Mamba)
DATA_SOURCE  = "btxrd"
BTXRD_DIR    = "/kaggle/input/datasets/nguyetanhmai/btxrd-split/BTXRD_split"                 # None = tự tìm (thư mục chứa train/ val/ test/); vd "/kaggle/input/datasets/nguyetanhmai/btxrd-split/BTXRD_split"
NNUNET_RAW_DIR = None               # dùng khi DATA_SOURCE="nnunet_raw", vd "/kaggle/input/umamba-data/Dataset704_Endovis17"
DATASET_ID   = 705
DATASET_NAME = f"Dataset{DATASET_ID:03d}_BTXRD"
LABEL_MODE   = "benign_malignant"   # "benign_malignant": 0 nền / 1 lành tính / 2 ác tính | "binary": nền / khối u
MAX_SIZE     = 1024                 # resize cạnh dài nhất về <= MAX_SIZE (ảnh X-quang gốc rất lớn); None = giữ nguyên

# ---------- Huấn luyện ----------
# Trainer dạng *Kaggle = trainer gốc + cho phép chỉnh epoch/iter qua biến môi trường (xem bước 4)
#   nnUNetTrainerSwinUMambaKaggle        : Swin-UMamba, pretrain ImageNet
#   nnUNetTrainerSwinUMambaDKaggle       : Swin-UMamba†, pretrain ImageNet
#   nnUNetTrainerSwinUMambaScratchKaggle : Swin-UMamba, không pretrain
#   nnUNetTrainerSwinUMambaDScratchKaggle: Swin-UMamba†, không pretrain
TRAINER         = "nnUNetTrainerSwinUMambaKaggle"
FOLD            = "0"               # BTXRD: fold 0 = đúng split train/val có sẵn (splits_final.json). "all" = train trên train+val
EPOCHS          = 50               # code gốc mặc định 1000 epoch -> quá dài cho Kaggle
ITERS_PER_EPOCH = 100
SAVE_EVERY      = 10                # lưu checkpoint_latest mỗi N epoch (để resume)
BATCH_SIZE      = 4              # None = để nnU-Net tự chọn; giảm (vd 4) nếu bị CUDA OOM
RESUME_DIR      = None              # vd "/kaggle/input/<output-phien-truoc>/Swin-UMamba/data/nnUNet_results"

# ---------- Cache môi trường ----------
WHEEL_DIR = "/kaggle/input/datasets/nguyetanhmai/setup-envi/swin_umamba_wheels"   # None = tự tìm file mamba_ssm-*.whl trong /kaggle/input; hoặc chỉ rõ, vd "/kaggle/input/swin-umamba-wheels"

PRETRAIN_URL = "https://github.com/MzeroMiko/VMamba/releases/download/%23v0cls/vssmtiny_dp01_ckpt_epoch_292.pth"

# ---------- Biến môi trường (được kế thừa bởi các lệnh `!`) ----------
os.environ.update({
    "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",   # torch>=2.6 mặc định weights_only=True -> không load được checkpoint nnU-Net
    "nnUNet_n_proc_DA": "4",                   # Kaggle có 4 CPU
    "nnUNet_compile": "f",
    "MPLBACKEND": "Agg",
    "SUM_EPOCHS": str(EPOCHS),
    "SUM_ITERS": str(ITERS_PER_EPOCH),
    "SUM_SAVE_EVERY": str(SAVE_EVERY),
})

## 1. Kiểm tra GPU & clone repo từ GitHub

In [2]:
!nvidia-smi
import subprocess
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=False)
!git -C {REPO} log --oneline -1

Tue Nov 17 13:43:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Cloning into '/kaggle/working/Swin-UMamba'...


2657c9d (grafted, HEAD -> main, origin/main, origin/HEAD) split_data


## 2. Cài môi trường (có cache wheel)
Bước chậm nhất là build `causal-conv1d` và `mamba-ssm` (biên dịch CUDA, có thể mất 20–40 phút). Ô dưới đây xử lý như sau:
- **Lần đầu:** build hai gói này thành file wheel và lưu vào `/kaggle/working/swin_umamba_wheels/`, kèm file `env_tag.txt` ghi phiên bản torch/CUDA/Python.
- **Các lần sau:** notebook tự tìm file `mamba_ssm-*.whl` và `causal_conv1d-*.whl` ở bất kỳ đâu trong `/kaggle/input` (hoặc trong `WHEEL_DIR` nếu bạn đặt), cài thẳng từ wheel rồi chạy thử kernel CUDA. Nếu wheel không còn tương thích (Kaggle đã đổi phiên bản torch), notebook tự build lại.

**Cách lưu cache để dùng lại** (chỉ cần làm 1 lần, sau khi lần chạy đầu thành công):
1. *Save Version* để lưu notebook, mở version vừa lưu, vào tab **Output**, chọn **New Dataset**, đặt tên ví dụ `swin-umamba-wheels`.
2. Ở các lần chạy sau, *Add Input* thêm dataset đó vào notebook. Cách upload nào cũng được: cả thư mục, hay chỉ 3 file `.whl` và `env_tag.txt`.
3. Nên bật *Settings → Environment → **Pin to original environment***. Kaggle cập nhật image định kỳ; nếu phiên bản torch đổi, wheel cũ không dùng được nữa và notebook sẽ tự build lại.

Các thư viện thuần Python còn lại (nnU-Net, monai, timm, …) vẫn cài từ PyPI như bình thường, chỉ mất khoảng 1–2 phút.

In [3]:
import sys, torch
ENV_TAG = f"torch{torch.__version__.split('+')[0]}-cu{torch.version.cuda}-py{sys.version_info.major}{sys.version_info.minor}"
print("ENV_TAG:", ENV_TAG)

def find_wheel_cache():
    """Tìm mọi thư mục chứa đủ 2 wheel (trong WHEEL_DIR, /kaggle/input hoặc /kaggle/working), ưu tiên thư mục có env_tag khớp."""
    roots = [WHEEL_DIR] if WHEEL_DIR else []
    roots += sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/mamba_ssm-*.whl", recursive=True)})
    roots += [f"{WORK}/swin_umamba_wheels"]
    cands = []
    for d in roots:
        if not (d and glob.glob(f"{d}/mamba_ssm-*.whl") and glob.glob(f"{d}/causal_conv1d-*.whl")):
            continue
        tag_file = f"{d}/env_tag.txt"
        tag = open(tag_file).read().strip() if os.path.isfile(tag_file) else "?"
        print(f"  tìm thấy wheel: {d}  (env_tag={tag})")
        cands.append((tag != ENV_TAG, d))
    return sorted(cands)[0][1] if cands else None

def mamba_ok():
    test = ("import torch; from mamba_ssm.ops.selective_scan_interface import selective_scan_fn; "
            "u=torch.randn(1,4,16,device='cuda'); A=-torch.rand(4,8,device='cuda'); B=torch.randn(1,8,16,device='cuda'); "
            "selective_scan_fn(u,u,A,B,B)")
    return subprocess.run(["python", "-c", test]).returncode == 0

%cd {REPO}/swin_umamba
!pip install -q -e .
# setup.py ghim monai==1.3.0 (không tương thích numpy 2 trên Kaggle) -> dùng 1.4.0, API các block vẫn giữ nguyên
!pip install -q "monai==1.4.0" timm torchinfo einops numba packaging ninja

WHEEL_CACHE = find_wheel_cache()
ok = False
if WHEEL_CACHE:
    print("Cài mamba từ cache:", WHEEL_CACHE)
    !pip install -q --no-deps --force-reinstall {WHEEL_CACHE}/causal_conv1d-*.whl {WHEEL_CACHE}/mamba_ssm-*.whl
    ok = mamba_ok()   # wheel build với torch khác phiên bản sẽ lỗi ở đây -> tự build lại
    print("cache dùng được" if ok else "cache KHÔNG tương thích với torch hiện tại -> build lại")
if not ok:
    print("Không có cache khớp -> build wheel (lần đầu sẽ lâu)...")
    WHEEL_OUT = f"{WORK}/swin_umamba_wheels"
    !rm -rf {WHEEL_OUT} && mkdir -p {WHEEL_OUT}
    !pip wheel -q --no-build-isolation --no-deps causal-conv1d mamba-ssm -w {WHEEL_OUT}
    !pip install -q --no-deps {WHEEL_OUT}/causal_conv1d-*.whl {WHEEL_OUT}/mamba_ssm-*.whl
    open(f"{WHEEL_OUT}/env_tag.txt", "w").write(ENV_TAG)
    !ls -la {WHEEL_OUT}
%cd {REPO}

ENV_TAG: torch2.11.0-cu13.0-py313
/kaggle/working/Swin-UMamba/swin_umamba
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for nnunetv2 (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
imagecodecs 2026.11.11 requires numpy>=2.1, but you have numpy 1.26.4 which

In [4]:
# Kiểm tra kernel CUDA selective_scan của mamba
check = ("import torch, mamba_ssm; "
         "from mamba_ssm.ops.selective_scan_interface import selective_scan_fn; "
         "u=torch.randn(1,4,16,device='cuda'); d=torch.randn(1,4,16,device='cuda'); A=-torch.rand(4,8,device='cuda'); "
         "B=torch.randn(1,8,16,device='cuda'); C=torch.randn(1,8,16,device='cuda'); "
         "print('torch', torch.__version__, '| mamba_ssm', mamba_ssm.__version__, '| out', selective_scan_fn(u,d,A,B,C).shape)")
subprocess.run(["python", "-c", check], check=True)
!python -c "import nnunetv2, monai; from nnunetv2.paths import nnUNet_raw; print('nnUNet_raw =', nnUNet_raw)"

torch 2.11.0+cu130 | mamba_ssm 2.3.2.post1 | out torch.Size([1, 4, 16])
nnUNet_raw = /kaggle/working/Swin-UMamba/data/nnUNet_raw


## 3. Tải pretrained VMamba-Tiny (ImageNet) & chuẩn bị thư mục dữ liệu

In [5]:
os.chdir(REPO)
ckpt = f"{REPO}/data/pretrained/vmamba/vmamba_tiny_e292.pth"
os.makedirs(os.path.dirname(ckpt), exist_ok=True)
import urllib.request
# file hợp lệ ~87 MB; file rỗng/hỏng (do lần tải trước bị lỗi) thì xoá và tải lại
if os.path.isfile(ckpt) and os.path.getsize(ckpt) < 50 * 2**20:
    os.remove(ckpt)
if not os.path.isfile(ckpt):
    urllib.request.urlretrieve(PRETRAIN_URL, ckpt + ".part")   # lỗi HTTP (404...) sẽ báo ngay, không tạo file rỗng
    os.rename(ckpt + ".part", ckpt)
assert os.path.getsize(ckpt) > 50 * 2**20, "File pretrained không hợp lệ"
print("pretrained:", ckpt, os.path.getsize(ckpt) // 2**20, "MB")

# raw & preprocessed nằm ở /tmp (symlink vào repo/data), results nằm trong /kaggle/working (được lưu lại)
RAW_DS = f"{TMP}/nnUNet_raw/{DATASET_NAME}"
os.makedirs(RAW_DS, exist_ok=True)
os.makedirs(f"{TMP}/nnUNet_preprocessed", exist_ok=True)
os.makedirs(f"{REPO}/data/nnUNet_results", exist_ok=True)
for link, target in [(f"{REPO}/data/nnUNet_raw/{DATASET_NAME}", RAW_DS),
                     (f"{REPO}/data/nnUNet_preprocessed", f"{TMP}/nnUNet_preprocessed")]:
    if not os.path.lexists(link):
        os.symlink(target, link)

if RESUME_DIR:   # khôi phục kết quả phiên trước để train tiếp
    !cp -rn {RESUME_DIR}/. {REPO}/data/nnUNet_results/
    print("restored results from", RESUME_DIR)

pretrained: /kaggle/working/Swin-UMamba/data/pretrained/vmamba/vmamba_tiny_e292.pth 87 MB


## 4. Trainer cho Kaggle
Các trainer này kế thừa nguyên trainer gốc (cùng AdamW, lr 1e-4, cosine schedule, đóng băng encoder 10 epoch đầu, deep supervision), chỉ thay đổi:
- `num_epochs`, `num_iterations_per_epoch`, `save_every`: đọc từ biến môi trường;
- không lưu `checkpoint_{epoch}.pth` (mỗi file khoảng 0.5 GB, dễ làm đầy 20 GB của `/kaggle/working`). Vẫn giữ `checkpoint_latest`, `checkpoint_best` và `checkpoint_final`.

In [6]:
trainer_src = '''
import os
import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMamba import nnUNetTrainerSwinUMamba
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaD import nnUNetTrainerSwinUMambaD
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaScratch import nnUNetTrainerSwinUMambaScratch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaDScratch import nnUNetTrainerSwinUMambaDScratch


def _kaggle_setup(self):
    self.num_epochs = int(os.environ.get("SUM_EPOCHS", 100))
    self.num_iterations_per_epoch = int(os.environ.get("SUM_ITERS", 250))
    self.save_every = int(os.environ.get("SUM_SAVE_EVERY", 10))

'''
for base in ["nnUNetTrainerSwinUMamba", "nnUNetTrainerSwinUMambaD",
             "nnUNetTrainerSwinUMambaScratch", "nnUNetTrainerSwinUMambaDScratch"]:
    trainer_src += f'''
class {base}Kaggle({base}):
    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict,
                 unpack_dataset: bool = True, device: torch.device = torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, unpack_dataset, device)
        _kaggle_setup(self)

    def on_epoch_end(self):
        nnUNetTrainer.on_epoch_end(self)   # bỏ qua việc lưu checkpoint_{{epoch}}.pth của lớp cha

'''
path = f"{REPO}/swin_umamba/nnunetv2/training/nnUNetTrainer/nnUNetTrainerSwinUMambaKaggle.py"
open(path, "w").write(trainer_src)
!python -c "from nnunetv2.training.nnUNetTrainer.nnUNetTrainerSwinUMambaKaggle import *; print('OK')"

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/kaggle/working/Swin-UMamba/swin_umamba/nnunetv2/training/nnUNetTrainer/nnUNetTrainerSwinUMambaD.py:18: SyntaxWarning: invalid escape sequence '\d'
  """ Swin-UMamba$\dagger$ with Mamba-based decoder"""
/kaggle/working/Swin-UMamba/swin_umamba/nnunetv2/training/nnUNetTrainer/nnUNetTrainerSwinUMambaDScratch.py:19: SyntaxWarning: invalid escape sequence '\d'
  """ Swin-UMamba$\dagger$ with Mamba-based decoder (trained from scratch) """
OK


## 5. Chuẩn bị dữ liệu theo định dạng nnU-Net
**BTXRD (đã chia train/val/test):** chỉ dùng các ảnh có nhãn **polygon** (vùng khối u); bỏ qua `rectangle` (bbox).
- Lớp của khối u lấy từ cột `type` trong `split.csv` (benign/malignant); nếu ảnh không có trong csv thì suy ra từ tên nhãn (`osteosarcoma`, `other mt`, `*sarcoma` → ác tính, còn lại → lành tính).
- Mask PNG: 0 = nền, 1 = lành tính, 2 = ác tính.
- `train` + `val` → `imagesTr/labelsTr`; `test` → `imagesTs/labelsTs`. Danh sách train/val được ghi ra `splits_final.json` (ở bước 6) để fold 0 của nnU-Net validate đúng trên tập `val`.

In [7]:
convert_src = r"""
import os, sys, json, glob, csv
import numpy as np, cv2
from PIL import Image, ImageOps

src, out, meta, mode, max_size, name = sys.argv[1:7]
max_size = int(max_size) if max_size != "None" else None

# lớp ảnh (benign/malignant) từ split.csv nếu có
types = {}
for p in [os.path.join(src, "split.csv"), os.path.join(os.path.dirname(src), "split.csv")]:
    if os.path.isfile(p):
        for r in csv.DictReader(open(p, encoding="utf-8-sig")):
            if r.get("image_id") and r.get("type"):
                types[os.path.splitext(r["image_id"].strip())[0]] = r["type"].strip().lower()
        print("split.csv:", p, "|", len(types), "ảnh có type")
        break

MALIGNANT = {"osteosarcoma", "other mt", "ewing sarcoma", "chondrosarcoma"}
def is_malignant(stem, label):
    if stem in types:
        return types[stem] == "malignant"
    l = label.strip().lower()
    return l in MALIGNANT or "sarcoma" in l

if mode == "binary":
    labels = {"background": 0, "tumor": 1}
    cls = lambda stem, l: 1
else:
    labels = {"background": 0, "benign": 1, "malignant": 2}
    cls = lambda stem, l: 2 if is_malignant(stem, l) else 1

for d in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    os.makedirs(os.path.join(out, d), exist_ok=True)
os.makedirs(meta, exist_ok=True)

ids = {"train": [], "val": [], "test": []}
info = []
for split in ["train", "val", "test"]:
    sdir = os.path.join(src, split)
    assert os.path.isdir(os.path.join(sdir, "images")), f"Thiếu {sdir}/images (có: {os.listdir(src)})"
    imgs = {os.path.splitext(f)[0]: os.path.join(sdir, "images", f) for f in os.listdir(os.path.join(sdir, "images"))}
    sub = "Ts" if split == "test" else "Tr"
    skipped = 0
    for jf in sorted(glob.glob(os.path.join(sdir, "Annotations", "*.json"))):
        stem = os.path.splitext(os.path.basename(jf))[0]
        ann = json.load(open(jf, encoding="utf-8"))
        polys = [s for s in ann["shapes"] if s.get("shape_type") == "polygon" and len(s["points"]) >= 3]
        if stem not in imgs or not polys:
            skipped += 1
            continue
        img = ImageOps.exif_transpose(Image.open(imgs[stem])).convert("L")   # LabelMe cũng áp dụng EXIF orientation
        w, h = img.size
        sx = w / float(ann.get("imageWidth") or w)
        sy = h / float(ann.get("imageHeight") or h)
        mask = np.zeros((h, w), np.uint8)
        for c in (1, 2):   # vẽ ác tính sau để ưu tiên khi chồng lấn
            for s in polys:
                if cls(stem, s["label"]) == c:
                    pts = np.array([[x * sx, y * sy] for x, y in s["points"]], np.float64).round().astype(np.int32)
                    cv2.fillPoly(mask, [pts], c)
        img = np.array(img)
        if max_size and max(h, w) > max_size:
            r = max_size / max(h, w)
            nw, nh = int(round(w * r)), int(round(h * r))
            img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
            mask = cv2.resize(mask, (nw, nh), interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(os.path.join(out, f"images{sub}", f"{stem}_0000.png"), img)
        cv2.imwrite(os.path.join(out, f"labels{sub}", f"{stem}.png"), mask)
        ids[split].append(stem)
        info.append((stem, split, "malignant" if mask.max() == 2 else "benign", "|".join(sorted({s["label"] for s in polys}))))
    print(f"{split}: {len(ids[split])} ảnh (bỏ qua {skipped} ảnh không có polygon/ảnh)")

json.dump({"channel_names": {"0": "X-ray"}, "labels": labels, "numTraining": len(ids["train"]) + len(ids["val"]),
           "file_ending": ".png", "name": name,
           "description": f"BTXRD bone tumor X-ray, polygons -> segmentation masks ({mode})"},
          open(os.path.join(out, "dataset.json"), "w"), indent=2)
# split train/val cố định cho nnU-Net (fold 0)
json.dump([{"train": sorted(ids["train"]), "val": sorted(ids["val"])}], open(os.path.join(meta, "splits_final.json"), "w"), indent=2)
with open(os.path.join(meta, "cases.csv"), "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(["case", "split", "type", "labels"]); wr.writerows(info)
print("labels:", labels)
"""
open(f"{TMP}/convert_btxrd.py", "w").write(convert_src)
META = f"{TMP}/btxrd_meta"   # splits_final.json + cases.csv (lớp ảnh, dùng khi đánh giá)

if DATA_SOURCE == "btxrd":
    if BTXRD_DIR is None:
        # thư mục gốc chứa train/val/test, mỗi tập có images/ + Annotations/
        cands = glob.glob("/kaggle/input/**/train/Annotations", recursive=True) + glob.glob(f"{REPO}/data/**/train/Annotations", recursive=True)
        found = [os.path.dirname(os.path.dirname(p)) for p in cands
                 if all(os.path.isdir(os.path.join(os.path.dirname(os.path.dirname(p)), s, "images")) for s in ["train", "val", "test"])]
        assert found, "Không tìm thấy thư mục BTXRD dạng train/ val/ test/ (mỗi tập gồm images/ + Annotations/)"
        BTXRD_DIR = found[0]
    print("BTXRD_DIR =", BTXRD_DIR, "->", sorted(os.listdir(BTXRD_DIR)))
    # chuyển lại nếu chưa có hoặc LABEL_MODE đã đổi
    ds_json = f"{RAW_DS}/dataset.json"
    n_labels = 3 if LABEL_MODE == "benign_malignant" else 2
    if not (os.path.isfile(ds_json) and len(json.load(open(ds_json))["labels"]) == n_labels and os.path.isfile(f"{META}/splits_final.json")):
        !rm -rf {RAW_DS}/* {META}
        !python {TMP}/convert_btxrd.py "{BTXRD_DIR}" "{RAW_DS}" "{META}" {LABEL_MODE} {MAX_SIZE} {DATASET_NAME}
else:
    # Dữ liệu đã ở định dạng nnU-Net (vd Dataset704_Endovis17 của U-Mamba): DATASET_ID/NAME phải khớp tên thư mục
    !cp -rn {NNUNET_RAW_DIR}/. {RAW_DS}/

!cat {RAW_DS}/dataset.json
for d in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    print(d, len(glob.glob(f"{RAW_DS}/{d}/*")))

BTXRD_DIR = /kaggle/input/datasets/nguyetanhmai/btxrd-split/BTXRD_split -> ['dataset.xlsx', 'split.csv', 'test', 'train', 'val']
split.csv: /kaggle/input/datasets/nguyetanhmai/btxrd-split/BTXRD_split/split.csv | 1867 ảnh có type
train: 1494 ảnh (bỏ qua 0 ảnh không có polygon/ảnh)
val: 280 ảnh (bỏ qua 0 ảnh không có polygon/ảnh)
test: 93 ảnh (bỏ qua 0 ảnh không có polygon/ảnh)
labels: {'background': 0, 'benign': 1, 'malignant': 2}
{
  "channel_names": {
    "0": "X-ray"
  },
  "labels": {
    "background": 0,
    "benign": 1,
    "malignant": 2
  },
  "numTraining": 1774,
  "file_ending": ".png",
  "name": "Dataset705_BTXRD",
  "description": "BTXRD bone tumor X-ray, polygons -> segmentation masks (benign_malignant)"
}imagesTr 1774
labelsTr 1774
imagesTs 93
labelsTs 93


## 6. Plan & preprocess (giống README gốc)

In [8]:
os.chdir(REPO)
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} -c 2d -np 4 --verify_dataset_integrity

# split train/val cố định -> fold 0 của nnU-Net validate đúng trên tập val
if DATA_SOURCE == "btxrd":
    !cp {META}/splits_final.json {TMP}/nnUNet_preprocessed/{DATASET_NAME}/splits_final.json
    sp = json.load(open(f"{TMP}/nnUNet_preprocessed/{DATASET_NAME}/splits_final.json"))
    print("splits_final.json: train", len(sp[0]["train"]), "| val", len(sp[0]["val"]))
    if FOLD not in ("0", "all"):
        print("CẢNH BÁO: splits_final.json chỉ có fold 0 -> FOLD khác 0 sẽ bị nnU-Net chia ngẫu nhiên 80/20")

plans_path = f"{TMP}/nnUNet_preprocessed/{DATASET_NAME}/nnUNetPlans.json"
plans = json.load(open(plans_path))
if BATCH_SIZE:
    plans["configurations"]["2d"]["batch_size"] = int(BATCH_SIZE)
    json.dump(plans, open(plans_path, "w"), indent=2)
cfg = plans["configurations"]["2d"]
print("patch_size:", cfg["patch_size"], "| batch_size:", cfg["batch_size"])

Fingerprint extraction...
Dataset705_BTXRD
Using <class 'nnunetv2.imageio.natural_image_reader_writer.NaturalImage2DIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.natural_image_reader_writer.NaturalImage2DIO'> as reader/writer
100%|███████████████████████████████████████| 1774/1774 [00:35<00:00, 49.41it/s]
Experiment planning...
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 13, 'patch_size': array([640, 384]), 'median_image_size_in_voxels': array([1024.,  716.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization'], 'use_mask_for_norm': [False], 'UNet_class_name': 'PlainConvUNet', 'UNet_base_num_features': 32, 'n_conv_per_stage_encoder': (2, 2, 2, 2, 2, 2, 2), 'n_conv_per_stage_decoder': (2, 2, 2, 2, 2, 2), 'num_pool_per_axis': [6, 6], 

## 7. Train + Validation
- Gọi `nnUNetv2_train <ID> 2d <FOLD> -tr <TRAINER>` giống script gốc.
- Khi train xong, nnU-Net tự chạy **validation** trên fold val và ghi kết quả vào `fold_X/validation/summary.json`.
- Nếu đã có `checkpoint_latest.pth` (phiên trước bị dừng, hoặc đã đặt `RESUME_DIR`), notebook sẽ tự thêm `--c` để train tiếp.

In [ ]:
os.chdir(REPO)
OUT_DIR  = f"{REPO}/data/nnUNet_results/{DATASET_NAME}/{TRAINER}__nnUNetPlans__2d"
FOLD_DIR = f"{OUT_DIR}/fold_{FOLD}"
resume = "--c" if os.path.isfile(f"{FOLD_DIR}/checkpoint_latest.pth") or os.path.isfile(f"{FOLD_DIR}/checkpoint_final.pth") else ""
print("resume flag:", resume or "(train mới)")
!CUDA_VISIBLE_DEVICES=0 nnUNetv2_train {DATASET_ID} 2d {FOLD} -tr {TRAINER} {resume}

resume flag: (train mới)
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Using device: cuda:0
/kaggle/working/Swin-UMamba/swin_umamba/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.grad_scaler = GradScaler() if self.device.type == 'cuda' else None

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
######################################################

In [ ]:
# Kết quả validation & đường cong huấn luyện
import json
from IPython.display import Image as IPImage, display
summ = f"{FOLD_DIR}/validation/summary.json"
if os.path.isfile(summ):
    s = json.load(open(summ))
    print("Validation mean Dice (foreground):", round(s["foreground_mean"]["Dice"], 4))
    CLASS_NAMES = {v: k for k, v in json.load(open(f"{RAW_DS}/dataset.json"))["labels"].items()}
    for k, v in s["mean"].items():
        print(f"  class {k} ({CLASS_NAMES.get(int(k), '?')}): Dice={v['Dice']:.4f}  IoU={v['IoU']:.4f}")
else:
    print("Chưa có validation/summary.json (train chưa xong?)")
if os.path.isfile(f"{FOLD_DIR}/progress.png"):
    display(IPImage(f"{FOLD_DIR}/progress.png"))

## 8. Inference trên tập test (giống script gốc: `--disable_tta`)

In [ ]:
os.chdir(REPO)
CKPT = next(c for c in ["checkpoint_final.pth", "checkpoint_best.pth", "checkpoint_latest.pth"]
            if os.path.isfile(f"{FOLD_DIR}/{c}"))
PRED_DIR = f"{OUT_DIR}/pred_results_fold{FOLD}"
print("checkpoint:", CKPT)
!CUDA_VISIBLE_DEVICES=0 nnUNetv2_predict -i "{RAW_DS}/imagesTs" -o "{PRED_DIR}" -d {DATASET_ID} -c 2d -tr {TRAINER} -f {FOLD} -chk {CKPT} --disable_tta

## 9. Đánh giá DSC & NSD (dùng script đánh giá gốc cho ảnh PNG 2D)

In [ ]:
os.chdir(REPO)
!python evaluation/endoscopy_DSC_Eval.py --gt_path "{RAW_DS}/labelsTs" --seg_path "{PRED_DIR}" --save_path "{OUT_DIR}/metric_DSC_fold{FOLD}.csv"
!python evaluation/endoscopy_NSD_Eval.py --gt_path "{RAW_DS}/labelsTs" --seg_path "{PRED_DIR}" --save_path "{OUT_DIR}/metric_NSD_fold{FOLD}.csv"

### 9b. Dice theo từng lớp & độ chính xác phân loại lành tính / ác tính
Lớp dự đoán của mỗi ảnh = lớp (lành/ác) chiếm nhiều pixel nhất trong mask dự đoán; nếu không có pixel u nào → `none`.

In [ ]:
import numpy as np, cv2, pandas as pd
CLASS_NAMES = {v: k for k, v in json.load(open(f"{RAW_DS}/dataset.json"))["labels"].items()}
fg = [c for c in sorted(CLASS_NAMES) if c > 0]

def dice(a, b):
    s = a.sum() + b.sum()
    return np.nan if s == 0 else 2.0 * (a & b).sum() / s

rows = []
for n in sorted(f for f in os.listdir(PRED_DIR) if f.endswith(".png")):
    gt = cv2.imread(f"{RAW_DS}/labelsTs/{n}", cv2.IMREAD_UNCHANGED)
    pr = cv2.imread(f"{PRED_DIR}/{n}", cv2.IMREAD_UNCHANGED)
    r = {"case": n[:-4], "Dice_tumor": dice(gt > 0, pr > 0)}
    for c in fg:
        r[f"Dice_{CLASS_NAMES[c]}"] = dice(gt == c, pr == c)
    area = {c: int((pr == c).sum()) for c in fg}
    gt_area = {c: int((gt == c).sum()) for c in fg}
    r["gt_class"] = CLASS_NAMES[max(gt_area, key=gt_area.get)]
    r["pred_class"] = CLASS_NAMES[max(area, key=area.get)] if sum(area.values()) else "none"
    rows.append(r)
df = pd.DataFrame(rows)
df.to_csv(f"{OUT_DIR}/metric_class_fold{FOLD}.csv", index=False)
pd.set_option("display.width", 200)
print(df.round(4).to_string(index=False))
print("\nMean Dice (bỏ qua ảnh không có lớp đó ở cả GT và dự đoán):")
print(df.filter(like="Dice_").mean().round(4).to_string())
if len(fg) > 1:
    acc = (df.gt_class == df.pred_class).mean()
    print(f"\nĐộ chính xác phân loại lành/ác (theo ảnh): {acc:.4f}  ({(df.gt_class == df.pred_class).sum()}/{len(df)})")
    print(pd.crosstab(df.gt_class, df.pred_class, rownames=["GT"], colnames=["Pred"]))

## 10. Trực quan hoá vài kết quả

In [ ]:
import numpy as np, cv2, random
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
CMAP = ListedColormap(["black", "lime", "red"])   # 1 = lành tính (xanh), 2 = ác tính (đỏ)
%matplotlib inline

names = sorted(os.listdir(PRED_DIR))
names = [n for n in names if n.endswith(".png")]
random.Random(0).shuffle(names)
k = min(4, len(names))
fig, ax = plt.subplots(k, 3, figsize=(12, 4 * k), squeeze=False)
for i, n in enumerate(names[:k]):
    img = cv2.imread(f"{RAW_DS}/imagesTs/{n[:-4]}_0000.png", cv2.IMREAD_GRAYSCALE)
    gt  = cv2.imread(f"{RAW_DS}/labelsTs/{n}", cv2.IMREAD_UNCHANGED)
    pr  = cv2.imread(f"{PRED_DIR}/{n}", cv2.IMREAD_UNCHANGED)
    ax[i, 0].imshow(img, cmap="gray"); ax[i, 0].set_title(n)
    ax[i, 1].imshow(img, cmap="gray"); ax[i, 1].imshow(np.ma.masked_equal(gt, 0), alpha=0.5, cmap=CMAP, vmin=0, vmax=2, interpolation="nearest"); ax[i, 1].set_title("Ground truth")
    ax[i, 2].imshow(img, cmap="gray"); ax[i, 2].imshow(np.ma.masked_equal(pr, 0), alpha=0.5, cmap=CMAP, vmin=0, vmax=2, interpolation="nearest"); ax[i, 2].set_title("Prediction")
    for a in ax[i]: a.axis("off")
fig.legend(handles=[Patch(color="lime", label="lành tính (1)"), Patch(color="red", label="ác tính (2)")], loc="upper center", ncol=2)
plt.tight_layout(rect=(0, 0, 1, 0.97)); plt.show()

## 11. Lưu kết quả
Mọi thứ trong `Swin-UMamba/data/nnUNet_results/` (checkpoint, log, validation, dự đoán, file CSV metric) nằm trong `/kaggle/working` và sẽ được lưu cùng version notebook. Để train tiếp ở phiên sau: thêm output của version này làm input, rồi đặt `RESUME_DIR` trỏ tới thư mục `nnUNet_results` trong đó.

In [ ]:
!du -sh {REPO}/data/nnUNet_results
!ls -la {FOLD_DIR}